In [1]:
import sys
print(sys.executable)

c:\IIIT\RAG\venv\Scripts\python.exe


In [1]:
import pydantic
import langchain
import transformers

print("✅ Environment clean")

c:\IIIT\RAG\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Environment clean


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline

from langchain.chains import RetrievalQA

from transformers import pipeline

In [3]:
loader1 = PyPDFLoader("C:\\IIIT\\RAG\\data\\AST-1.pdf")
loader2 = PyPDFLoader("C:\\IIIT\\RAG\\data\\AST-2.pdf")

docs = loader1.load() + loader2.load()

print("✅ Documents loaded:", len(docs))

✅ Documents loaded: 11


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

print("✅ Chunks created:", len(chunks))

✅ Chunks created: 42


In [5]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embeddings ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

c:\IIIT\RAG\venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embeddings ready


In [6]:
vector_db = FAISS.from_documents(chunks, embedding)

print("✅ Vector DB created")

✅ Vector DB created


In [20]:
retriever_basic = vector_db.as_retriever(search_kwargs={"k": 5})

retriever_mmr = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "lambda_mult": 0.7}
)

print("✅ Retrievers ready")

✅ Retrievers ready


In [23]:
pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=512
)

llm = HuggingFacePipeline(pipeline=pipe)

print("✅ LLM ready")

c:\IIIT\RAG\venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ LLM ready


In [31]:
from langchain.prompts import PromptTemplate

prompt_template = """
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Provide a detailed and well-explained answer.
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [32]:
qa_basic = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever_basic,
    chain_type_kwargs={"prompt": PROMPT}
)

qa_mmr = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever_mmr,
    chain_type_kwargs={"prompt": PROMPT}
)

print("✅ RAG ready")

✅ RAG ready


In [ ]:
questions = [
    "What is modularization in machine learning?",
    "Explain pipeline.py",
    "What is pytest?",
    "What are types of testing?",
    "What is packaging in machine learning?"
]

for q in questions:
    print("\n❓ Question:", q)
    print("🔹 Basic:", qa_basic.run(q))
    print("🔸 MMR:", qa_mmr.run(q))


❓ Question: What is modularization in machine learning?
🔹 Basic: Converting a ML model from research environment format (Notebook) to product environment format (.py files) Understanding the functionality of each file and folder
🔸 MMR: Transitioning from Research to Production Environment

❓ Question: Explain pipeline.py
🔹 Basic: pipeline.py is a Python program that enables the user to build a pipeline using transformation classes and modeling algorithm.
🔸 MMR: pipeline.py is a collection of Python modules, and it is a useful way for us to publish related functionality so that different project applications can install our package and make use of the Python modules.

❓ Question: What is pytest?
🔹 Basic: a popular testing framework in Python that simplifies the process of writing and executing tests
🔸 MMR: a popular testing framework in Python

❓ Question: What are types of testing?
🔹 Basic: Unit test, Integration test and system test
🔸 MMR: Unit test, Integration test and system test


Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\Anoop\AppData\Local\Programs\Python\Python310\lib\asyncio\events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\Anoop\AppData\Local\Programs\Python\Python310\lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\Anoop\AppData\Local\Programs\Python\Python310\lib\asyncio\events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\Anoop\AppData\Local\Programs\Py

In [35]:
import gradio as gr

# Function to rebuild pipeline based on chunk size
def build_rag(chunk_size):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=100
    )
    
    chunks = splitter.split_documents(docs)
    
    vector_db = FAISS.from_documents(chunks, embedding)
    
    retriever_basic = vector_db.as_retriever(search_kwargs={"k": 5})
    retriever_mmr = vector_db.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 5, "lambda_mult": 0.7}
    )
    
    qa_basic = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever_basic,
    chain_type_kwargs={"prompt": PROMPT}
    )
    
    qa_mmr = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever_mmr,
        chain_type_kwargs={"prompt": PROMPT}
    )
    
    return qa_basic, qa_mmr


# Main function for UI
def compare(query, chunk_size):
    qa_basic, qa_mmr = build_rag(chunk_size)
    
    basic = qa_basic.run(query)
    mmr = qa_mmr.run(query)
    
    return basic, mmr


# Gradio UI
app = gr.Interface(
    fn=compare,
    inputs=[
        gr.Textbox(label="Enter your question"),
        gr.Dropdown(
            choices=[300, 500, 800],
            value=500,
            label="Select Chunk Size"
        )
    ],
    outputs=[
        gr.Textbox(label="Basic RAG Output"),
        gr.Textbox(label="MMR Output")
    ],
    title="RAG vs MMR with Chunk Size Comparison"
)

app.launch()

Running on local URL:  http://127.0.0.1:7866

To create a public link, set `share=True` in `launch()`.


IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------


Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\Anoop\AppData\Local\Programs\Python\Python310\lib\asyncio\events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\Anoop\AppData\Local\Programs\Python\Python310\lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
